In [2]:
!pip install "unsloth[gemma4]"
!pip install trl transformers accelerate datasets bitsandbytes

In [10]:
import torch
torch.cuda.empty_cache()

In [1]:
from unsloth import FastLanguageModel


max_seq_length = 2048 # Supports up to 8192, but 2048 saves VRAM
dtype = None          # None for auto-detection (Float16 for T4, Bfloat16 for A100)
load_in_4bit = True   # Use 4-bit quantization to fit on a free T4 GPU

# 1. Load Gemma 4 Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it", # Replace with E2B-it or 12B-it if needed
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Set up Fast LoRA weights
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,               # Keep low for 100 samples to prevent overfitting
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,     # Optimized to 0 by Unsloth
    bias = "none",        # Optimized to none
    use_gradient_checkpointing = "unsloth", # Saves massive VRAM
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

[unsloth.import_fixes|WARNING]Unsloth: torch==2.11.0a0+eb65b36914.nv26.2 requires torchvision>=0.26.0, but found torchvision==0.25.0a0+1e53952f.nv26.2.44259020. Try updating torchvision via `pip install --upgrade "torchvision>=0.26.0"`. Please refer to https://pytorch.org/get-started/previous-versions/ for more information.
Detected a custom/source build. Continuing with a warning. Set UNSLOTH_SKIP_TORCHVISION_CHECK=1 to silence this.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[bitsandbytes.cextension|WARNING]No prebuilt binary for CUDA 13.1, loading CUDA 13.0 instead. Set BNB_CUDA_VERSION to override.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.18: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB MIG 3g.20gb. Num GPUs = 1. Max memory: 19.625 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0a0+eb65b36914.nv26.02. CUDA: 8.0. CUDA Toolkit: 13.1. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: `flash_attention_2` is not supported for `gemma4` because max attention head dim 512 exceeds the Flash Attention 2 limit of 256 - defaulting to `sdpa`.


Loading weights: 100%|██████████| 2130/2130 [00:03<00:00, 583.23it/s]


Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [2]:
samples = [
    
     # --- Math ---
    
    {
        "messages": [
            {"role": "user", "content": "What is 2 + 2?"},
            {"role": "assistant", "content": "4"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is 9 - 3?"},
            {"role": "assistant", "content": "6"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is 7 x 8?"},
            {"role": "assistant", "content": "56"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is 100 / 4?"},
            {"role": "assistant", "content": "25"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is 15 + 27?"},
            {"role": "assistant", "content": "42"},
        ]
    },

    # --- Yes/No ---
    {
        "messages": [
            {"role": "user", "content": "Is the sky blue?"},
            {"role": "assistant", "content": "Yes"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Is a spider an insect?"},
            {"role": "assistant", "content": "No"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Is water wet?"},
            {"role": "assistant", "content": "Yes"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Is the sun a planet?"},
            {"role": "assistant", "content": "No"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Do plants need sunlight?"},
            {"role": "assistant", "content": "Yes"},
        ]
    },

    # --- Definition ---
    {
        "messages": [
            {"role": "user", "content": "What does 'photosynthesis' mean?"},
            {"role": "assistant", "content": "The process by which plants convert sunlight into energy."},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What does 'gravity' mean?"},
            {"role": "assistant", "content": "The force that pulls objects toward one another."},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What does 'ecosystem' mean?"},
            {"role": "assistant", "content": "A community of living organisms interacting with their environment."},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What does 'inflation' mean?"},
            {"role": "assistant", "content": "A general increase in prices over time."},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What does 'democracy' mean?"},
            {"role": "assistant", "content": "A system of government where power lies with the people."},
        ]
    },
    # --- True/False ---
    {
        "messages": [
            {"role": "user", "content": "The Earth is flat."},
            {"role": "assistant", "content": "False."},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Water boils at 100 degrees Celsius at sea level."},
            {"role": "assistant", "content": "True."},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Humans have three lungs."},
            {"role": "assistant", "content": "False."},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "The Great Wall of China is visible from space with the naked eye."},
            {"role": "assistant", "content": "False."},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Sharks are fish."},
            {"role": "assistant", "content": "True."},
        ]
    },

    # --- Capital city ---
    {
        "messages": [
            {"role": "user", "content": "What is the capital of France?"},
            {"role": "assistant", "content": "Paris"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is the capital of Japan?"},
            {"role": "assistant", "content": "Tokyo"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is the capital of Canada?"},
            {"role": "assistant", "content": "TOttawa"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is the capital of Australia?"},
            {"role": "assistant", "content": "Canberra"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is the capital of Egypt?"},
            {"role": "assistant", "content": "Cairo"},
        ]
    },

    # --- Multiple choice ---
    {
        "messages": [
            {"role": "user", "content": "Which is a fruit? A) Carrot B) Apple C) Potato"},
            {"role": "assistant", "content": "B)"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Which is a mammal? A) Shark B) Frog C) Dolphin"},
            {"role": "assistant", "content": "C)"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Which is a planet? A) Moon B) Mars C) Sun"},
            {"role": "assistant", "content": "B)"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Which is a primary color? A) Green B) Purple C) Blue"},
            {"role": "assistant", "content": "C)"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Which is a metal? A) Oxygen B) Iron C) Nitrogen"},
            {"role": "assistant", "content": "B)"},
        ]
    },

    # --- Unit conversion ---
    {
        "messages": [
            {"role": "user", "content": "How many centimeters are in 1 meter?"},
            {"role": "assistant", "content": "100 centimeters"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How many grams are in 1 kilogram?"},
            {"role": "assistant", "content": "1000 grams"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How many minutes are in 2 hours?"},
            {"role": "assistant", "content": "120 minutes"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How many milliliters are in 1 liter?"},
            {"role": "assistant", "content": "T1000 milliliters"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How many inches are in 1 foot?"},
            {"role": "assistant", "content": "12 inches"},
        ]
    },

    # --- Spelling ---
    {
        "messages": [
            {"role": "user", "content": "How do you spell 'necessary'?"},
            {"role": "assistant", "content": "N-E-C-E-S-S-A-R-Y"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How do you spell 'rhythm'?"},
            {"role": "assistant", "content": "R-H-Y-T-H-M"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How do you spell 'occasion'?"},
            {"role": "assistant", "content": "O-C-C-A-S-I-O-N"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How do you spell 'separate'?"},
            {"role": "assistant", "content": "S-E-P-A-R-A-T-E"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How do you spell 'definitely'?"},
            {"role": "assistant", "content": "D-E-F-I-N-I-T-E-L-Y"},
        ]
    },
    # --- Translation ---
    {
        "messages": [
            {"role": "user", "content": "How do you say 'hello' in Spanish?"},
            {"role": "assistant", "content": "Hola"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How do you say 'thank you' in French?"},
            {"role": "assistant", "content": "Merci"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How do you say 'goodbye' in Italian?"},
            {"role": "assistant", "content": "Arrivederci"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How do you say 'friend' in German?"},
            {"role": "assistant", "content": "Freund"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How do you say 'water' in Japanese?"},
            {"role": "assistant", "content": "Mizu"},
        ]
    },
    # --- Synonym ---
    {
        "messages": [
            {"role": "user", "content": "What is a synonym for 'happy'?"},
            {"role": "assistant", "content": "Joyful"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a synonym for 'fast'?"},
            {"role": "assistant", "content": "Quick"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a synonym for 'smart'?"},
            {"role": "assistant", "content": "Intelligen"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a synonym for 'big'?"},
            {"role": "assistant", "content": "Large"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a synonym for 'sad'?"},
            {"role": "assistant", "content": "Unhappy"},
        ]
    },
    # --- Antonym ---
    {
        "messages": [
            {"role": "user", "content": "What is the antonym of 'hot'?"},
            {"role": "assistant", "content": "Cold"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is the antonym of 'up'?"},
            {"role": "assistant", "content": "Down"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is the antonym of 'light'?"},
            {"role": "assistant", "content": "Dark"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is the antonym of 'fast'?"},
            {"role": "assistant", "content": "Slow"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is the antonym of 'happy'?"},
            {"role": "assistant", "content": "Sad"},
        ]
    },
    # --- Color ---
    {
        "messages": [
            {"role": "user", "content": "What color is a banana?"},
            {"role": "assistant", "content": "Yellow"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What color is grass?"},
            {"role": "assistant", "content": "Green"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What color is the sky on a clear day?"},
            {"role": "assistant", "content": "Blue"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What color is a stop sign?"},
            {"role": "assistant", "content": "Red"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What color is coal?"},
            {"role": "assistant", "content": "Black"},
        ]
    },
    # --- Counting ---
    {
        "messages": [
            {"role": "user", "content": "How many legs does a spider have?"},
            {"role": "assistant", "content": "8"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How many sides does a hexagon have?"},
            {"role": "assistant", "content": "6"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How many continents are there?"},
            {"role": "assistant", "content": "7"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How many days are in a week?"},
            {"role": "assistant", "content": "7"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How many strings does a standard guitar have?"},
            {"role": "assistant", "content": "6"},
        ]
    },
    # --- Comparison ---
    {
        "messages": [
            {"role": "user", "content": "Which is bigger, an elephant or a mouse?"},
            {"role": "assistant", "content": "Elephant"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Which is heavier, a feather or a brick?"},
            {"role": "assistant", "content": "Brick"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Which is taller, a giraffe or a horse?"},
            {"role": "assistant", "content": "Giraffe"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Which is faster, a cheetah or a turtle?"},
            {"role": "assistant", "content": "Cheetah"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Which is colder, ice or boiling water?"},
            {"role": "assistant", "content": "Ice"},
        ]
    },
    # --- Rhyme ---
    {
        "messages": [
            {"role": "user", "content": "What rhymes with 'cat'?"},
            {"role": "assistant", "content": "hat"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What rhymes with 'tree'?"},
            {"role": "assistant", "content": "bee"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What rhymes with 'light'?"},
            {"role": "assistant", "content": "night"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What rhymes with 'car'?"},
            {"role": "assistant", "content": "star"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What rhymes with 'blue'?"},
            {"role": "assistant", "content": "true"},
        ]
    },
    # --- Fill in the blank ---
    {
        "messages": [
            {"role": "user", "content": "Complete: The sun rises in the ___."},
            {"role": "assistant", "content": "'east'"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Complete: A baby dog is called a ___."},
            {"role": "assistant", "content": "'puppy'"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Complete: Water freezes into ___."},
            {"role": "assistant", "content": "'ice'"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Complete: Bees make ___."},
            {"role": "assistant", "content": "'honey'"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Complete: The opposite of day is ___."},
            {"role": "assistant", "content": "'night'"},
        ]
    },

    # --- Percentage ---
    {
        "messages": [
            {"role": "user", "content": "What is 50% of 200?"},
            {"role": "assistant", "content": "100"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is 10% of 50?"},
            {"role": "assistant", "content": "5"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is 25% of 80?"},
            {"role": "assistant", "content": "20"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is 20% of 150?"},
            {"role": "assistant", "content": "30"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is 75% of 40?"},
            {"role": "assistant", "content": "30"},
        ]
    },
    # --- Time calculation ---
    {
        "messages": [
            {"role": "user", "content": "If it's 3:00 PM now, what time is it in 2 hours?"},
            {"role": "assistant", "content": "5:00 PM"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "If a movie starts at 7:00 PM and lasts 90 minutes, when does it end?"},
            {"role": "assistant", "content": "8:30 PM"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "If it's 11:30 AM now, what time was it 45 minutes ago?"},
            {"role": "assistant", "content": "10:45 AM"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "If a flight departs at 6:15 AM and takes 3 hours, when does it land?"},
            {"role": "assistant", "content": "9:15 AM"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "If it's midnight now, what time was it 6 hours ago?"},
            {"role": "assistant", "content": "6:00 PM"},
        ]
    },
    # --- List ---
    {
        "messages": [
            {"role": "user", "content": "Name three primary colors."},
            {"role": "assistant", "content": "Red, blue, green"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Name three planets closest to the sun."},
            {"role": "assistant", "content": "Mercury, Venus, Earth"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Name four fruits."},
            {"role": "assistant", "content": "Apple, banana, orange, grapes"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Name five continents."},
            {"role": "assistant", "content": "Asia, Africa, Europe, North America, South America"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Name three types of weather."},
            {"role": "assistant", "content": "Rain, snow, sunshine"},
        ]
    },
    # --- Date ---
    {
        "messages": [
            {"role": "user", "content": "What year did World War II end?"},
            {"role": "assistant", "content": "1945"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What year did the moon landing happen?"},
            {"role": "assistant", "content": "1969"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What year was the Declaration of Independence signed?"},
            {"role": "assistant", "content": "1776"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What year did the Berlin Wall fall?"},
            {"role": "assistant", "content": "1989"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What year was the first iPhone released?"},
            {"role": "assistant", "content": "2007"},
        ]
    }
]

In [3]:
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
dataset = Dataset.from_list(samples)
# 2. Set Gemma 4 Chat Template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-4",  # Match the exact Gemma 4 template
)


def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}


dataset = dataset.map(formatting_prompts_func, batched=True)

# 3. Configure Trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        num_train_epochs=4,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

# 4. Correctly mask instruction part using Gemma 4 turn delimiters
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",  # Gemma 4 instruction delimiter
    response_part="<|turn>model\n",  # Gemma 4 model output delimiter
)

# Retrain the model
trainer_stats = trainer.train()

Map: 100%|██████████| 100/100 [00:00<00:00, 2589.62 examples/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: We found double BOS tokens - we shall remove one automatically.


Map: 100%|██████████| 100/100 [00:00<00:00, 16787.29 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 4 | Total steps = 52
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 36,700,160 of 8,032,856,608 (0.46% trained)
Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss
1,8.222610
2,7.454525
3,7.459607
4,8.079183
5,5.872706
6,5.531029
7,5.964383
8,2.645555
9,0.980886
10,0.937770


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-52/tokenizer_config.json.


In [4]:
# Save locally to Colab storage
model.save_pretrained("gemma4_lora_model")
tokenizer.save_pretrained("gemma4_lora_model")

# Optional: Push to Hugging Face Hub (uncomment below if needed)
# model.push_to_hub("your_username/gemma4_lora_model", token = "YOUR_HF_TOKEN")
# tokenizer.push_to_hub("your_username/gemma4_lora_model", token = "YOUR_HF_TOKEN")


Unsloth: Restored added_tokens_decoder metadata in gemma4_lora_model/tokenizer_config.json.


['gemma4_lora_model/processor_config.json']

In [1]:
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load your local fine-tuned LoRA weights
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "gemma4_lora_model", # Your local folder path
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
# Enable native fast 2x inference speed
FastLanguageModel.for_inference(model)

[unsloth.import_fixes|WARNING]Unsloth: torch==2.11.0a0+eb65b36914.nv26.2 requires torchvision>=0.26.0, but found torchvision==0.25.0a0+1e53952f.nv26.2.44259020. Try updating torchvision via `pip install --upgrade "torchvision>=0.26.0"`. Please refer to https://pytorch.org/get-started/previous-versions/ for more information.
Detected a custom/source build. Continuing with a warning. Set UNSLOTH_SKIP_TORCHVISION_CHECK=1 to silence this.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[bitsandbytes.cextension|WARNING]No prebuilt binary for CUDA 13.1, loading CUDA 13.0 instead. Set BNB_CUDA_VERSION to override.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.18: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB MIG 3g.20gb. Num GPUs = 1. Max memory: 19.625 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0a0+eb65b36914.nv26.02. CUDA: 8.0. CUDA Toolkit: 13.1. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: `flash_attention_2` is not supported for `gemma4` because max attention head dim 512 exceeds the Flash Attention 2 limit of 256 - defaulting to `sdpa`.


Loading weights: 100%|██████████| 2130/2130 [00:03<00:00, 573.36it/s] 


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma4ForConditionalGeneration(
      (model): Gemma4Model(
        (language_model): Gemma4TextModel(
          (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
          (layers): ModuleList(
            (0): Gemma4TextDecoderLayer(
              (self_attn): Gemma4TextAttention(
                (q_norm): Gemma4RMSNorm()
                (k_norm): Gemma4RMSNorm()
                (v_norm): Gemma4RMSNorm()
                (k_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=2560, out_features=512, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=2560, out_features=16, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=16, out_features=512, bias

In [ ]:
import torch
from transformers import TextStreamer

# 1. Use a custom streamer to clean up structural tokens on the fly
class CleanTextStreamer(TextStreamer):
    def on_finalized_text(self, text: str, stream_end: bool = False):
        # Prevent outputting raw structure if the model outputs JSON format
        cleaned_text = text.replace("[{'type': 'text', 'text': '", "").replace("'}]", "")
        super().on_finalized_text(cleaned_text, stream_end=stream_end)

streamer = CleanTextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

print("Gemma 4 Fine-Tuned Model ready! Type 'exit' to quit.\n")

# Keep track of history cleanly using plain text strings
messages = []

while True:
    user_prompt = input("User: ").strip()
    if user_prompt.lower() == "exit":
        print("Ending session.")
        break
    if not user_prompt:
        continue

    # Add user query in standard format
    messages.append({
        "role": "user",
        "content": [{"type": "text", "text": user_prompt}]
    })

    # Prepare input tensor
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    print("Model: ", end="", flush=True)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            streamer=streamer,
            max_new_tokens=512,
            use_cache=True,
            temperature=0.3,
            top_p=0.9,
        )

    # Extract clean text to append back into chat history
    input_len = inputs.shape[1]
    gen_tokens = outputs[0][input_len:]
    raw_response = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    # Clean string format before pushing to memory
    clean_response = raw_response.replace("[{'type': 'text', 'text': '", "").rstrip(" '}]")

    # Add back pure text content so the model doesn't loop on JSON syntax
    messages.append({
        "role": "assistant",
        "content": [{"type": "text", "text": clean_response}]
    })

    print("\n" + "-" * 50)

Gemma 4 Fine-Tuned Model ready! Type 'exit' to quit.



User:  Is the sky blue?


Model: 

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Yes

--------------------------------------------------


...

Model: I do not have information about a person named W Selvamurthy.

--------------------------------------------------


User:  Where is Amity University?


Model: India

--------------------------------------------------


User:  Where is Ritsomeikan?


Model: Japan

--------------------------------------------------
